In [2]:
import numpy as np
from pathlib import Path

# --- CONFIG ---
# Base 600-cell edge list: vertices 0..119, undirected edges (u, v)
BASE_EDGE_LIST_PATH = Path("nb02_SM8_600cell_chain_edgelist.txt")  # must be in this folder
OUTPUT_EDGE_LIST_PATH = Path("nb02_SM8_tiled600_2ring_edgelist.txt")

N_VERT = 120          # vertices per 600-cell
N_RING1 = 6           # first ring around center
N_RING2 = 12          # second ring around ring1

# Cell IDs:
# 0          : center
# 1..6       : first ring
# 7..18      : second ring
N_CELLS = 1 + N_RING1 + N_RING2  # 19

def load_base_edges(path: Path):
    edges = []
    with path.open("r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) != 2:
                continue
            u, v = map(int, parts)
            if u == v:
                continue
            edges.append((u, v))
    return edges

base_edges = load_base_edges(BASE_EDGE_LIST_PATH)
print(f"Loaded {len(base_edges)} base edges from {BASE_EDGE_LIST_PATH}")

def offset_vertex(cell_id: int, local_v: int) -> int:
    return cell_id * N_VERT + local_v

all_edges = []

# --- 1. Internal edges for each 600-cell copy ---
for cell_id in range(N_CELLS):
    for u, v in base_edges:
        U = offset_vertex(cell_id, u)
        V = offset_vertex(cell_id, v)
        if U != V:
            all_edges.append((U, V))

print(f"After replication: {len(all_edges)} edges (with multiplicity)")

# --- 2. Gluing pattern (graph-theoretic 2-ring) ---
# A. Center (0) glued to first ring (1..6) via identity vertex mapping
for ring1_id in range(1, 1 + N_RING1):
    for local_v in range(N_VERT):
        center_v = offset_vertex(0, local_v)
        neigh_v = offset_vertex(ring1_id, local_v)
        all_edges.append((center_v, neigh_v))

# B. Each first-ring cell glued to two distinct second-ring cells
#    ring1: 1..6, ring2: 7..18
for i in range(N_RING1):
    ring1_id = 1 + i
    ring2_a = 7 + 2 * i
    ring2_b = 7 + 2 * i + 1
    for local_v in range(N_VERT):
        v1 = offset_vertex(ring1_id, local_v)
        va = offset_vertex(ring2_a, local_v)
        vb = offset_vertex(ring2_b, local_v)
        all_edges.append((v1, va))
        all_edges.append((v1, vb))

print(f"After provisional 2-ring gluing: {len(all_edges)} edges (with multiplicity)")

# --- 3. Clean up: undirected, unique, sorted ---
clean_edges = set()
for u, v in all_edges:
    if u == v:
        continue
    if u > v:
        u, v = v, u
    clean_edges.add((u, v))

clean_edges = sorted(clean_edges)
print(f"Unique undirected edges: {len(clean_edges)}")
print(f"Total vertices: {N_CELLS * N_VERT}")

# --- 4. Write out ---
with OUTPUT_EDGE_LIST_PATH.open("w") as f:
    for u, v in clean_edges:
        f.write(f"{u} {v}\n")

print(f"Wrote 2-ring tiled 600-cell edge list to {OUTPUT_EDGE_LIST_PATH}")


Loaded 41880 base edges from nb02_SM8_600cell_chain_edgelist.txt
After replication: 795720 edges (with multiplicity)
After provisional 2-ring gluing: 797880 edges (with multiplicity)
Unique undirected edges: 59040
Total vertices: 2280
Wrote 2-ring tiled 600-cell edge list to nb02_SM8_tiled600_2ring_edgelist.txt
